# 🛰️ ICARUS_CORE — Central Research Orchestrator
## Multi-Domain Scientific Research Engine (Account 1)

**Version:** 0.2.0 (Distributed Architecture)  
**Role:** Central Orchestrator & Research Planner  
**LLM Engine:** GPT-OSS-120B (via Groq API)  
**Deployment Profile:** Kaggle Account 1 (CPU or GPU)

---

### Distributed 4-Account Deployment Topology

```
                         ┌────────────────────────────────────┐
                         │    Kaggle Account 1: ICARUS_CORE   │
                         │    - GPT-OSS-120B (Groq API)       │
                         │    - Registries & Health Checker   │
                         │    - Task Dispatcher (Retries)     │
                         │    - Multi-Domain Report Generator │
                         └─────────────────┬──────────────────┘
                                           │
                    ┌──────────────────────┼──────────────────────┐
                    │ HTTP/JSON            │ HTTP/JSON            │ HTTP/JSON
                    │ (ngrok tunnel)       │ (ngrok tunnel)       │ (ngrok tunnel)
                    ▼                      ▼                      ▼
        ┌──────────────────────┐┌──────────────────────┐┌──────────────────────┐
        │  Account 2: WORKER_EO ││  Account 3: WORKER_2 ││  Account 4: WORKER_3 │
        │  Earth Observation   ││  Atmosphere, Oceans  ││  Solid Earth, Space  │
        │  - Prithvi EO 2.0    ││  - Aurora, Waves     ││  - PhaseNet, Surya   │
        │  - Spectral (NDVI)   ││  - Prithvi-WxC       ││  - Lunar Crater, FITS│
        └──────────────────────┘└──────────────────────┘└──────────────────────┘
```


---
## 1. Environment Detection & Kaggle Secrets

Inspect the runtime environment and securely retrieve API credentials (`GROQ_API_KEY`, `NGROK_AUTH_TOKEN`) from Kaggle Secrets without hardcoding.


In [ ]:
# ============================================================
# 1. Setup, Environment & Secrets
# ============================================================
import os
import sys
import json
import time
from pathlib import Path

# Detect Kaggle Environment
IS_KAGGLE = os.path.exists('/kaggle') or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

def get_secret(key: str, default: str = "") -> str:
    """Fetches secret from Kaggle Secrets or environment variables."""
    if IS_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            user_secrets = UserSecretsClient()
            return user_secrets.get_secret(key)
        except Exception:
            pass
    return os.environ.get(key, default)

GROQ_API_KEY = get_secret("GROQ_API_KEY")
NGROK_AUTH_TOKEN = get_secret("NGROK_AUTH_TOKEN")

print("=" * 60)
print("🖥️  ICARUS_CORE Runtime Initialization")
print("=" * 60)
print(f"  Platform:        {'Kaggle Notebook' if IS_KAGGLE else 'Local / Server'}")
print(f"  Python Version:  {sys.version.split()[0]}")
print(f"  GROQ_API_KEY:    {'✅ Configured' if GROQ_API_KEY else '⚠️ Missing (Will use deterministic planner)'}")
print(f"  NGROK_TOKEN:     {'✅ Configured' if NGROK_AUTH_TOKEN else '⚠️ Missing (Optional on Orchestrator)'}")
print("=" * 60)

---
## 2. Core Library Imports

Import the shared ICARUS schemas, registries, dispatcher, and orchestrator components.


In [ ]:
# ============================================================
# 2. Core ICARUS Imports
# ============================================================
try:
    import icarus
except ImportError:
    # If running directly inside Kaggle, ensure repo directory is in sys.path
    repo_path = Path("/kaggle/working/ICARUS").resolve()
    if repo_path.exists():
        sys.path.insert(0, str(repo_path))
    else:
        sys.path.insert(0, str(Path(".").resolve()))

from icarus.schemas import (
    DomainEnum,
    CapabilityStatus,
    TaskStatus,
    WorkerRuntimeStatus,
    SpatialBounds,
    TemporalRange,
    ResearchRequest,
    MultiDomainRequest,
    ResearchResponse,
    MultiDomainResponse,
    CapabilityDescriptor,
    WorkerCapabilitiesResponse,
    HealthCheckResponse,
    WorkerRegistration,
)
from icarus.core.worker_registry import WorkerRegistry
from icarus.core.health_checker import HealthChecker
from icarus.core.dispatcher import TaskDispatcher
from icarus.orchestrator.llm_planner import LLMPlanner, StructuredResearchPlan
from icarus.orchestrator.report_generator import ReportGenerator
from icarus.orchestrator.orchestrator import CentralResearchOrchestrator

print("✅ ICARUS core and orchestrator modules loaded successfully.")

---
## 3. Scientific Registries

Load the three machine-readable registries:
- **`model_registry.json`**: 13 specialist models with VRAM footprints and licenses.
- **`capability_registry.json`**: 50 scientific capabilities adhering to the 5-tier status taxonomy.
- **`dataset_registry.json`**: 23 authoritative scientific data providers.


In [ ]:
# ============================================================
# 3. Load Registries
# ============================================================
class LocalRegistryLoader:
    def __init__(self, filename: str):
        self.filename = filename
        self.data = self._load()

    def _load(self):
        p = Path(self.filename)
        if not p.exists():
            # Check Kaggle working directory
            p = Path("/kaggle/working/ICARUS") / self.filename
        if p.exists():
            with open(p, "r", encoding="utf-8") as f:
                return json.load(f)
        return {}

    def get_capability(self, cap_id: str):
        for c in self.data.get("capabilities", []):
            if c["id"] == cap_id:
                return c
        return None

cap_reg_loader = LocalRegistryLoader("capability_registry.json")
mod_reg_loader = LocalRegistryLoader("model_registry.json")
ds_reg_loader = LocalRegistryLoader("dataset_registry.json")

caps = cap_reg_loader.data.get("capabilities", [])
models = mod_reg_loader.data.get("models", [])
datasets = ds_reg_loader.data.get("datasets", [])

print(f"📊 Registries Loaded:")
print(f"  Capabilities: {len(caps)}")
print(f"  Models:       {len(models)}")
print(f"  Data Sources: {len(datasets)}")

---
## 4. Worker Discovery & Dynamic Registration

Register the endpoints of the three distributed Kaggle worker nodes:
- **Worker EO** (Account 2): Earth Observation & Prithvi EO 2.0
- **Worker 2** (Account 3): Atmosphere, Oceans, Climate
- **Worker 3** (Account 4): Solid Earth, Heliophysics, Planetary, Astronomy

When a worker starts in its own Kaggle notebook, it displays a public ngrok tunnel URL (e.g. `https://xxxx.ngrok-free.app`).  
Paste or set the endpoints below.


In [ ]:
# ============================================================
# 4. Initialize Worker Registry & Register Endpoints
# ============================================================
worker_registry = WorkerRegistry(timeout_seconds=10.0)
health_checker = HealthChecker(worker_registry, timeout_seconds=5.0)

# Configure Worker URLs (From environment, Kaggle Secrets, or manual input)
WORKER_ENDPOINTS = {
    "ICARUS_WORKER_EO": os.environ.get("ICARUS_WORKER_EO_URL", ""),
    "ICARUS_WORKER_2":  os.environ.get("ICARUS_WORKER_2_URL", ""),
    "ICARUS_WORKER_3":  os.environ.get("ICARUS_WORKER_3_URL", ""),
}

# Example: If you have an active ngrok URL from Worker EO:
# WORKER_ENDPOINTS["ICARUS_WORKER_EO"] = "https://your-ngrok-url.ngrok-free.app"

for name, url in WORKER_ENDPOINTS.items():
    if url:
        success, msg, _ = worker_registry.register_worker_endpoint(url, name=name)
        print(f"Registration [{name}]: {msg}")
    else:
        print(f"Notice: {name} URL not configured. Register when worker is online.")

# Health check probe
print("\n🩺 Probing Worker Health Status:")
health_results = health_checker.check_all_workers()
if not health_results:
    print("  No workers currently registered. Tasks will return UNAVAILABLE status until workers join.")
else:
    for wid, h in health_results.items():
        w = worker_registry.get_worker(wid)
        status_icon = "✅" if h and h.status == "healthy" else "⚠️"
        latency_str = f"{w.latency_ms}ms" if w and w.latency_ms else "N/A"
        print(f"  {status_icon} Worker: {w.name:20s} ID: {wid[:8]} | Status: {w.runtime_status} | Latency: {latency_str}")

---
## 5. Central Research Orchestrator Initialization

Instantiate the `CentralResearchOrchestrator` combining:
- **`LLMPlanner`**: Powered by `openai/gpt-oss-120b` via Groq
- **`TaskDispatcher`**: With automatic exponential backoff retries and timeouts
- **`ReportGenerator`**: Publication-quality cross-domain scientific synthesis


In [ ]:
# ============================================================
# 5. Initialize Central Orchestrator
# ============================================================
orchestrator = CentralResearchOrchestrator(
    worker_registry=worker_registry,
    health_checker=health_checker,
    capability_registry=cap_reg_loader,
    model_registry=mod_reg_loader,
    dataset_registry=ds_reg_loader,
    groq_api_key=GROQ_API_KEY,
)

def run_distributed_research(question: str, parameters: dict = None):
    """Top-level user interface for distributed scientific inquiries."""
    print(f"\n{'='*70}")
    print(f"🔬 ICARUS Research Query: {question}")
    print(f"{'='*70}")

    report, multi_resp, plan = orchestrator.execute_research(question, parameters=parameters)

    print(f"\n📋 1. Structured Research Plan:")
    print(f"   Identified Domains: {[d if isinstance(d, str) else d.value for d in plan.identified_domains]}")
    print(f"   Execution Strategy: {plan.execution_strategy}")
    print(f"   Rationale:          {plan.scientific_rationale}")
    print(f"   Subtasks ({len(plan.subtasks)}):")
    for i, st in enumerate(plan.subtasks, 1):
        print(f"     {i}. [{st.domain}] Task: {st.task} | Models: {st.required_models or 'Standard Algorithm'}")

    print(f"\n⚡ 2. Worker Dispatch Execution:")
    print(f"   Overall Status: {multi_resp.status}")
    for r in multi_resp.subtask_responses:
        status_ico = "✅" if r.status == "success" else ("⚠️" if r.status == "unavailable" else "❌")
        print(f"   {status_ico} Subtask [{r.domain}]: {r.status} (Worker: {r.worker_id[:8]})")

    print(f"\n📑 3. Scientific Report Synthesis:")
    print(report["markdown_report"])

    return report, multi_resp, plan

print("✅ Central Research Orchestrator is ready to accept scientific research questions.")

---
## 6. Demonstration Workflows

Run sample cross-domain and single-domain scientific inquiries.


In [ ]:
# ============================================================
# Demo 1: Multi-Domain Research Query
# ============================================================
query_1 = "Analyze the impact of severe drought and temperature warming trends on agricultural vegetation cover"
report_1, resp_1, plan_1 = run_distributed_research(query_1)

In [ ]:
# ============================================================
# Demo 2: Single-Domain Specialist Query
# ============================================================
query_2 = "Assess recent earthquake magnitudes and seismic event distributions along active fault lines"
report_2, resp_2, plan_2 = run_distributed_research(query_2)

---
## 7. Kaggle 4-Account Deployment Checklist

To operate ICARUS across your 4 Kaggle accounts:

1. **Start Workers First**:
   - Open **Account 2** (`ICARUS_WORKER_EO.ipynb`), **Account 3** (`ICARUS_WORKER_2.ipynb`), and **Account 4** (`ICARUS_WORKER_3.ipynb`).
   - Run all cells to launch the FastAPI services and start the ngrok tunnels.
   - Copy the public ngrok URLs displayed in the worker notebooks.
2. **Configure Account 1 (This Notebook)**:
   - Paste the URLs into the `WORKER_ENDPOINTS` dictionary in Section 4.
   - Add your `GROQ_API_KEY` to **Add-ons > Secrets**.
3. **Execute Scientific Research**:
   - Call `run_distributed_research("Your question")`.
   - GPT-OSS-120B decomposes the problem, routes subtasks to the appropriate GPU workers, and compiles a comprehensive report.
4. **VRAM Eviction**:
   - To free GPU memory on any worker, call:
     ```python
     orchestrator.unload_model(worker_id="worker_id", model_id="model_id")
     ```
